## 02 — Feature Extraction
### SIR-EEGNet | EEG-based Alzheimer's Disease Classification
### Extracts Relative Band Power (RBP) features from all 88 subjects.
### Saves a subject-keyed feature store to disk for notebooks 03-05.
### Feature vector per epoch: 19 channels × 5 bands = 95 dimensions

In [1]:
# %%
# Install dependencies

import subprocess, sys

def pip(*p):
    subprocess.run([sys.executable,'-m','pip','install',*p,'-q'], check=True)

pip('mne','awscli','scipy','numpy','pandas','tqdm')
print('Dependencies ready.')

Dependencies ready.


In [2]:
# %%
# Environment + paths

import os
from pathlib import Path

if os.path.exists('/content'):
    ENV, BASE = 'colab', Path('/content/sir-eegnet')
elif os.path.exists('/kaggle/working'):
    ENV, BASE = 'kaggle', Path('/kaggle/working/sir-eegnet')
else:
    ENV, BASE = 'local', Path.cwd().parent

DATA_DIR     = BASE / 'data'
DS_DIR       = DATA_DIR / 'ds004504'
DERIV_DIR    = DS_DIR / 'derivatives'
PARTICIPANTS = DS_DIR / 'participants.tsv'
FEATURES_DIR = BASE / 'features'

for d in [DATA_DIR, DS_DIR, DERIV_DIR, FEATURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Env: {ENV} | Features will be saved to: {FEATURES_DIR}')

Env: colab | Features will be saved to: /content/sir-eegnet/features


In [3]:
# %%
# Constants

import numpy as np
import pandas as pd
np.random.seed(42)

SFREQ          = 500.0
EPOCH_DURATION = 4.0
EPOCH_OVERLAP  = 0.5
EPOCH_SAMPLES  = int(EPOCH_DURATION * SFREQ)

GROUP_MAP   = {'A': 0, 'F': 1, 'C': 2}
LABEL_NAMES = {0: 'AD', 1: 'FTD', 2: 'HC'}

FREQ_BANDS = {
    'delta': (0.5, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta':  (13.0, 30.0),
    'gamma': (30.0, 45.0),
}

N_BANDS = len(FREQ_BANDS)
N_CH    = 19
N_FEAT  = N_CH * N_BANDS

print(f'Feature vector size: {N_CH} channels × {N_BANDS} bands = {N_FEAT} dimensions')

Feature vector size: 19 channels × 5 bands = 95 dimensions


In [4]:
# %%
# Download dataset (if needed)

def download_ds004504(deriv_dir, participants_path):
    if participants_path.exists() and len(list(deriv_dir.rglob('*.set'))) >= 80:
        print('Dataset already present. Skipping download.')
        return

    print('Downloading ds004504... (~2.7 GB)')

    r1 = subprocess.run([
        'aws','s3','sync',
        's3://openneuro.org/ds004504/derivatives/', str(deriv_dir),
        '--no-sign-request','--region','us-east-1'
    ], capture_output=True, text=True)

    if r1.returncode != 0:
        raise RuntimeError(r1.stderr[-600:])

    r2 = subprocess.run([
        'aws','s3','cp',
        's3://openneuro.org/ds004504/participants.tsv', str(participants_path),
        '--no-sign-request','--region','us-east-1'
    ], capture_output=True, text=True)

    if r2.returncode != 0:
        raise RuntimeError(r2.stderr[-600:])

    print('Download complete.')

download_ds004504(DERIV_DIR, PARTICIPANTS)

Download complete.


In [5]:
# %%
# Core functions

from scipy.signal import welch

def epoch_raw_signal(data, sfreq=500.0, duration=4.0, overlap=0.5):
    n_ch, n_t = data.shape
    ep_s   = int(duration * sfreq)
    step_s = int(ep_s * (1 - overlap))
    starts = range(0, n_t - ep_s + 1, step_s)
    return np.stack([data[:, s:s+ep_s] for s in starts], axis=0).astype(np.float32)


def compute_rbp(epoch, sfreq, freq_bands):
    nperseg    = min(256, epoch.shape[1])
    freqs, psd = welch(epoch, fs=sfreq, nperseg=nperseg, axis=-1)
    freq_res   = freqs[1] - freqs[0]

    abs_power  = np.stack([
        np.sum(psd[:, (freqs>=lo)&(freqs<=hi)], axis=-1) * freq_res
        for _, (lo, hi) in freq_bands.items()
    ], axis=0)

    total = abs_power.sum(axis=0, keepdims=True) + 1e-10
    rbp   = abs_power / total

    return rbp.T.flatten().astype(np.float32)


def compute_dtabr_per_channel(epoch, sfreq, freq_bands):
    nperseg    = min(256, epoch.shape[1])
    freqs, psd = welch(epoch, fs=sfreq, nperseg=nperseg, axis=-1)
    res        = freqs[1] - freqs[0]

    bp = {
        b: np.sum(psd[:,(freqs>=lo)&(freqs<=hi)],axis=-1)*res
        for b,(lo,hi) in freq_bands.items()
    }

    return ((bp['delta']+bp['theta']) / (bp['alpha']+bp['beta']+1e-10)).astype(np.float32)


print('Feature functions defined.')
print(f'RBP output size: {N_FEAT} per epoch')

Feature functions defined.
RBP output size: 95 per epoch


In [ ]:
# %%
# Feature extraction loop

import mne, pickle
mne.set_log_level('WARNING')

participants = pd.read_csv(PARTICIPANTS, sep='\t')
participants['label'] = participants['Group'].map(GROUP_MAP)
participants = participants.dropna(subset=['label'])
participants['label'] = participants['label'].astype(int)

subject_data = {}
failed       = []

print(f'Extracting features for {len(participants)} subjects...')
print()

for i, (_, row) in enumerate(participants.iterrows()):
    sid   = row['participant_id']
    label = int(row['label'])
    fpath = DERIV_DIR / sid / 'eeg' / f'{sid}_task-eyesclosed_eeg.set'

    if not fpath.exists():
        failed.append(sid)
        continue

    try:
        raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
        data   = raw.get_data()
        epochs = epoch_raw_signal(data, SFREQ, EPOCH_DURATION, EPOCH_OVERLAP)

        rbp_epochs   = np.stack([compute_rbp(ep, SFREQ, FREQ_BANDS) for ep in epochs])
        dtabr_epochs = np.stack([compute_dtabr_per_channel(ep, SFREQ, FREQ_BANDS) for ep in epochs])

        subject_data[sid] = {
            'epochs_raw': epochs,
            'rbp': rbp_epochs,
            'dtabr': dtabr_epochs,
            'label': label,
            'group': row['Group'],
            'mmse': float(row.get('MMSE', np.nan)),
            'age': float(row['Age']),
            'n_epochs': len(epochs),
        }

        if (i+1) % 15 == 0:
            print(f'  {i+1}/{len(participants)} done...')

    except Exception as e:
        print(f'  [FAIL] {sid}: {e}')
        failed.append(sid)

print(f'\nExtraction complete.')
print(f'Subjects loaded : {len(subject_data)}')
print(f'Failed          : {len(failed)}')

Extracting features for 88 subjects...



/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), prel

  15/88 done...


/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), prel

  30/88 done...


...

/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), prel

  75/88 done...


/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), preload=True, verbose=True)
/tmp/ipykernel_3894/229596212.py:28: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(fpath), prel


Extraction complete.
Subjects loaded : 88
Failed          : 0


: 